# Code Review Eval Guide

## Introduction

Code Reviews are an essential part of ensuring production code is tested, secure, and defect-free. AI coding tools like Codex can generate high-quality code reviews, and are rapidly augmenting - and even replacing - human code reviews. How can we prove that our AI code reviews are performing well?

This guide demonstrates how to build Evals for AI code review systems, starting with a simple system and evolving to use more sophisticated techniques:
  - Level 1: Simple Benchmark Eval
  - Level 2: Pairwise Comparison Eval
  - Level 3: Multi-turn Optimization Loop

Moving from "vibe reviewing" to evals enables faster experimentation and adoption of new models and reasoning levels, token usage optimization, and a higher level of trust in AI systems throughout the SDLC. 

TODO: Add links here to the example repo/notebook once ready

## Foundations

In order to construct a basic eval for code reviews, we're going to need the following pieces:
- a "golden dataset"
- a code reviewer model, the "system under test"
- a grader model, the "LLM-as-a-judge"
- a score aggregator

### The Golden Dataset

In the context of evals, a "golden dataset" is a trusted set of examples with expected answers/judgements that are used to measure model behavior. At a minimum, it should have:
1. **Inputs**: prompts, tasks, test cases, etc. 
2. **Outputs**: the correct answer, ideal behavior, annotataion, etc.

For this code review eval, we will start by using existing code reviews in the [openai/codex](https://github.com/openai/codex), the open-source repository for [Codex](https://openai.com/codex/). We can use the [GitHub CLI](https://cli.github.com) to retrieve a set of closed PRs with review comments. 

Thus, our golden dataset has the following:
1. **Inputs**: diff, code files modified, review comments
2. **Outputs**: review success (`True` if the PR was merged, `False` if it was not merged)

This dataset has enough for us to work on our eval, but we'll see how it can be improved as we evolve the example. 

### Code Reviewer Model

This is the "system under test", and is the model that we will be measuring and (soon) optimizing. 

In this simple example, we'll use `gpt-5.4` with the following prompt:

```markdown
You are an LLM code reviewer evaluating a pull request. Your job is to identify the most important issues in the diff, following the repository-specific instructions in AGENTS.md.

Output requirements:
- Return valid JSON only.
- Do not wrap the JSON in markdown fences.
- Follow this exact schema:

{
  "summary": "string",
  "comments": [
    {
      "path": "string",
      "line": 0,
      "severity": "low | medium | high",
      "category": "correctness | bug_risk | security | performance | testing | maintainability",
      "claim": "string",
      "suggestion": "string",
      "confidence": 0.0
    }
  ]
}
```

For now, we'll use an empty `AGENTS.md` file to focus on establishing a benchmark.

### Grader Model

This is the LLM which will be used to evaluate the output of the Code Reviewer model. Selecting an optimal grading model is beyond the scope of this guide. We will use `gpt-5.4-nano`, a low-latency model that is performant for classification tasks with clear instructions. We'll use the following prompt for grading:

```markdown
You are grading one LLM code review for a pull request.

Your task is to judge whether the generated review is acceptable overall.

Evaluation criteria:

1. Correctness
- Comments must be grounded in the actual diff.
- Comments must not invent nonexistent behavior, files, or risks.

2. Usefulness
- Comments should identify meaningful issues or omissions.
- Comments should be specific enough to help an engineer act.

3. Noise
- Penalize trivial style comments unless AGENTS.md explicitly prioritizes style.
- Penalize vague comments, redundant comments, or excessive low-value comments.
- A smaller set of strong comments is better than many weak comments.

4. Instruction adherence
- Judge the review against the provided AGENTS.md guidance while keeping correctness first.

How to use the human comments:
- Treat historical human comments as weak reference evidence, not absolute ground truth.
- They are useful context for whether the review noticed important concerns, but they are not the only valid source of truth.

Pass rule:
- Set `overall_pass` to `1` only when the review is acceptable overall for this pull request.
- A passing review should be grounded, useful, and not meaningfully noisy.

Output requirements:
- Return valid JSON only.
- Do not wrap the JSON in markdown fences.
- `noise=1` means the review is low-noise / acceptable on noise.
- Follow this exact schema:

{
  "correctness": 0,
  "usefulness": 0,
  "noise": 0,
  "overall_pass": 0,
  "reason": "string"
}

```

### Score Aggregator

We will use a simple average to aggregate the scores produced by the grader model:
```
pass_rate = sum(overall_pass) / pr_count
correctness_rate = sum(correctness) / pr_count
usefulness_rate = sum(usefulness) / pr_count
low_noise_rate = sum(noise) / pr_count
```

### The Eval Engine Loop

With all of our ingredients in place, the eval loop looks like:

```mermaid
flowchart TD
    A[Golden Data] --> | PR + Result | B[Reviewer Model]
    B --> | Code Review | C[Grader Model]        
    C --> A
    C --> |Score| D[Score Aggregator]
```



## Level 1: Simple Benchmark Eval

Now that we have our foundations in place, let's build the simple benchmark. You can find a reference implementation [here](evals/codereview_evals)

First, build the eval CLI:
```bash
cd examples/evals/codereview_evals
python -m venv .venv
source .venv/bin/activate
pip install -e .
```

### 1. Create Golden Dataset

The first step is to fetch PRs from `openai/codex`:

```bash
evalcr fetch-prs --repo openai/codex --limit 50
```

`evalcr fetch-prs` will gather PRs from GitHub and cache them locally. We will collect this data:
- PR metadata 
- merge state
- changed files
- unified diff
- issue comments
- inline review comments

### 2. Configure the Harness

We'll use the following files (in `1_benchmark_harness`) to configure our eval harness:
- [`eval_config.json`](evals/codereview_evals/1_benchmark_harness/eval_config.json): reviewer and grader model selection
- [`AGENTS.md`](evals/codereview_evals/1_benchmark_harness/AGENTS.md): code review guidelines
- [`reviewer_system.txt`](evals/codereview_evals/1_benchmark_harness/reviewer_system.txt): code reviewer system prompt
- [`review_output_schema.json`](evals/codereview_evals/1_benchmark_harness/review_output_schema.json) stores the structured reviewer output schema
- [`grader_system.txt`](evals/codereview_evals/1_benchmark_harness/grader_system.txt): grader system prompt
- [`grader_output_schema.json`](evals/codereview_evals/1_benchmark_harness/grader_output_schema.json) stores the structured grader output schema


### 3. Run the benchmark

```bash
evalcr benchmark run --type benchmark --cache-key openai_codex
```

Here's an example set of results from a selection of 50 PRs from `openai/codex`:
```json
{
  "total_examples": 50,
  "successful_examples": 48,
  "failed_examples": 2,
  "pass_rate": 0.3333333333333333,
  "correctness_rate": 0.5625,
  "usefulness_rate": 0.3125,
  "low_noise_rate": 0.7916666666666666,
  "merged_examples": 36,
  "closed_unmerged_examples": 12,
  "merged_pass_rate": 0.2777777777777778,
  "closed_unmerged_pass_rate": 0.5
}
```

The following command will also serve a full HTML report at localhost:8000:
```bash
evalcr visualize --run-id $RUN_ID # The RUN_ID is printed to the console by `evalcr benchmark run`
```


### 4. Interpreting the results

We can interpret the results from this summary in three parts:

First, check run reliability: `total_examples`, `successful_examples`, and `failed_examples` show whether the benchmark executed cleanly enough to trust the rest of the report. Failed examples usually indicate harness or runtime issues and should be separated from review-quality interpretation. The CSV and HTML report outputs show error messages for the failed runs, which can be useful for debugging harness issues.

Second, treat `pass_rate` as the strict overall quality bar, and use `correctness_rate`, `usefulness_rate`, and `low_noise_rate` to diagnose why the system passed or failed. These metrics answer different questions: whether feedback is right, whether it is actionable, and whether it avoids unnecessary comments. Read them together, not in isolation.

Third, use subgroup metrics such as merged versus closed-unmerged pass rate to identify where performance changes across PR types. Those splits are useful for spotting benchmark sensitivity and for finding cases where the review system performs better or worse than the aggregate score suggests.

## Level 3: Multi-turn Optimization Loop

Pairwise comparison is useful, but it can create a subtle failure mode: you can optimize toward a candidate that wins head-to-head while drifting away from a high absolute benchmark score. If Level 2 is your only objective, the system may learn to beat the current baseline pairwise while regressing on overall `pass_rate`.

Level 3 fixes that by making optimization explicitly multi-objective. At every step, we evaluate whether the candidate:
1. wins pairwise against the current champion
2. still achieves a strong standalone benchmark score

In the reference implementation, Level 3 lives in [`examples/evals/codereview_evals/3_optimization_harness`](evals/codereview_evals/3_optimization_harness). The loop keeps a **champion** and an editable **candidate**. Each iteration:
1. runs a pairwise comparison between champion and candidate
2. runs a Level 1 style benchmark on the candidate alone
3. computes a composite score from pairwise win rate and benchmark pass rate
4. applies a benchmark guardrail so pairwise gains do not come from a large benchmark regression
5. uses an optimizer model to revise the candidate `AGENTS.md` and reviewer system prompt for the next round

### 1. Configure the Optimization Harness

We'll use the following files in `3_optimization_harness`:
- [`eval_config.json`](evals/codereview_evals/3_optimization_harness/eval_config.json): reviewer, grader, and optimizer model selection plus loop defaults such as `max_steps`, `score_threshold`, and score weights
- [`AGENTS.md`](evals/codereview_evals/3_optimization_harness/AGENTS.md): optimization objective and guardrail framing
- [`baseline_AGENTS.md`](evals/codereview_evals/3_optimization_harness/baseline_AGENTS.md): initial champion policy
- [`baseline_reviewer_system.txt`](evals/codereview_evals/3_optimization_harness/baseline_reviewer_system.txt): initial champion reviewer prompt
- [`candidate_AGENTS.md`](evals/codereview_evals/3_optimization_harness/candidate_AGENTS.md): seed candidate policy
- [`candidate_reviewer_system.txt`](evals/codereview_evals/3_optimization_harness/candidate_reviewer_system.txt): seed candidate reviewer prompt
- [`benchmark_grader_system.txt`](evals/codereview_evals/3_optimization_harness/benchmark_grader_system.txt): Level 1 style benchmark judge prompt
- [`pairwise_judge_system.txt`](evals/codereview_evals/3_optimization_harness/pairwise_judge_system.txt): Level 2 style pairwise judge prompt
- [`optimizer_system.txt`](evals/codereview_evals/3_optimization_harness/optimizer_system.txt): instructions for proposing the next candidate revision
- [`review_output_schema.json`](evals/codereview_evals/3_optimization_harness/review_output_schema.json): structured review schema
- [`benchmark_grader_output_schema.json`](evals/codereview_evals/3_optimization_harness/benchmark_grader_output_schema.json): structured benchmark grading schema
- [`pairwise_output_schema.json`](evals/codereview_evals/3_optimization_harness/pairwise_output_schema.json): structured pairwise winner schema
- [`optimizer_output_schema.json`](evals/codereview_evals/3_optimization_harness/optimizer_output_schema.json): structured optimizer revision schema

### 2. Run the optimization loop

```bash
evalcr benchmark run --type optimizer --cache-key openai_codex
```

You can override the default stopping conditions from `eval_config.json`:

```bash
evalcr benchmark run --type optimizer --cache-key openai_codex --max-steps 4 --score-threshold 0.8
```

Each optimizer run writes into `3_optimization_harness/results/<run_id>/`:
- `iterations.json`: one row per optimization step
- `iterations.csv`: flattened table of the same optimization history
- `summary.json`: final stop reason, best step, and best scores
- `report.html`: top-level optimization history report
- `final_candidate_AGENTS.md`: the final champion policy text
- `final_candidate_reviewer_system.txt`: the final champion reviewer prompt
- `baseline/`: saved baseline benchmark artifacts
- `steps/step_XX/`: pairwise results, benchmark results, optimizer output, and the next candidate revision for each iteration

An example summary looks like this:
```json
{
  "steps_run": 3,
  "max_steps": 4,
  "score_threshold": 0.8,
  "stop_reason": "score_threshold_reached",
  "final_champion_label": "step_03_candidate",
  "final_champion_pass_rate": 0.68,
  "best_step": 3,
  "best_candidate_composite_score": 0.81,
  "best_candidate_win_rate": 0.86,
  "best_candidate_pass_rate": 0.76
}
```

### 3. Interpreting the results

The key idea in Level 3 is that the optimizer is no longer allowed to chase pairwise wins in isolation. The candidate needs to improve head-to-head while also preserving strong standalone review quality. That is why the stopping condition is based on a composite score and a benchmark guardrail, not on pairwise wins alone.

When you read `summary.json`, first look at `stop_reason`. If the loop stopped because it reached `score_threshold`, that means the candidate cleared the target objective under the current weighting. If it stopped because of `max_steps`, the optimizer failed to hit the target before the budget ran out, which usually means either the search space is too constrained or the optimizer prompt needs better evidence.

Next, compare `best_candidate_win_rate` and `best_candidate_pass_rate` together. A high pairwise win rate with a weak benchmark pass rate is exactly the failure mode Level 3 is designed to catch. A strong candidate should improve both or, at minimum, improve pairwise without violating the benchmark guardrail.

Finally, inspect the per-step artifacts in `steps/step_XX/`. Those files show why a step was promoted, what benchmark failures remained, and what the optimizer changed in the next candidate revision. This is the fastest way to determine whether the loop is making principled improvements or just moving instructions around without measurable benefit.
